<a href="https://colab.research.google.com/github/matti410/Trading-System-Creator_V4/blob/main/Trading_Grid_Search_v4_backtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ricerca di Trading System

Serve a **scoprire se un segnale di ingresso ha un vantaggio reale**, separando entry, uscita e "quando sto fermo" — cercarli insieme confonde quale dei tre sta davvero funzionando.

Si legge dall'alto in basso, senza celle facoltative: ogni sezione usa il risultato di quella prima.

**Ordine:** la Sessione 1 misura il segnale puro (nessun capitale, nessuna posizione). La Sessione 2 sceglie come uscirne e diventa un sistema vero, con trade e costi.

In [ ]:
!pip install -q TA-Lib backtesting

### Sezione 0 · Preparazione

Clona la repo e prepara l'ambiente. Va eseguita una volta sola a inizio sessione Colab.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
from pathlib import Path
from google.colab import userdata
import shutil
# Salva il token una volta in Colab: icona chiave a sinistra → "Secrets" → aggiungi GITHUB_TOKEN
# Assicurati che il nome del secret sia 'GITHUB_TOKEN' (o quello che preferisci) e non il token stesso.
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

REPO_DIR = Path('/content/Trading-System-Creator_V4')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)  # ripulisce il tentativo fallito precedente

!git clone https://{GITHUB_TOKEN}@github.com/matti410/Trading-System-Creator_V4.git

%cd Trading-System-Creator_V4

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import datetime as dt
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import talib as ta

import entry_long, entry_short
import exit_long, exit_short
import engine as eng
from engine.vwap_ops import vwap_anchored_daily
import engine.event_study
import engine.splitting
from engine.exit_search_bt import run_exit_search_bt

## Sezione 1 · I dati

Scarica le candele da MetaTrader 5.

**Parametri che puoi cambiare:** strumento, timeframe, periodo storico.

In [ ]:
symbol    = "EURUSD"
timeframe = 'M15'
FREQ      = "15min"      # come pandas scrive il timeframe: "15min", "h", "D"

df = pd.read_csv('EURUSD_M15.csv', index_col='Date')
df.index = pd.to_datetime(df.index, utc=True)

print(f"{symbol} {timeframe} — {len(df):,} barre  ·  da {df.index[0]}  a  {df.index[-1]}")
df.tail(3)

## Sezione 2 · Gli indicatori

Le condizioni di ingresso (`entry_long.py`, `entry_short.py`) non calcolano niente da sole: leggono colonne già pronte in questa tabella. Qui le costruiamo.

| Colonna | Cos'è | Chi la usa |
|---|---|---|
| `rsi` | oscillatore 0-100: sotto 30 il prezzo è "sceso troppo", sopra 70 "salito troppo" | l'entry mean reversion |
| `ema20`, `ema50` | medie mobili veloce e lenta | l'entry di controllo |
| `zlema50` | media mobile a ritardo ridotto | l'entry di trend |
| `atr`, `realized_vol` | quanto si muove il prezzo di solito | dimensionamento futuro |
| `vwap` | prezzo medio della giornata pesato per volume | i filtri di contesto |

Sulla **zero-lag EMA**: una media normale è sempre in ritardo; la ZLEMA lo compensa proiettando in avanti il prezzo, ma esagera i movimenti e genera più falsi attraversamenti — su M15 va bene (più campioni), ma è dove i costi di transazione pesano di più.

In [ ]:
df["rsi"] = ta.RSI(df["Close"], timeperiod=14)
df['macd'], df['macd_signal'], df['macd_hist'] = ta.MACD(df["Close"], fastperiod=12, slowperiod=26, signalperiod=9)
df["ema20"] = df["Close"].ewm(span=20, min_periods=20).mean()   # ← min_periods
df["ema50"] = df["Close"].ewm(span=50, min_periods=50).mean()   # ← min_periods
def zlema(series, period):
    """Media mobile a ritardo ridotto: proietta il prezzo per compensare il lag."""
    lag = int((period - 1) / 2)
    return (2 * series - series.shift(lag)).ewm(span=period, adjust=False).mean()

df["zlema50"] = zlema(df["Close"], 50)
df["atr"] = ta.ATR(df["High"], df["Low"], df["Close"], timeperiod=14)          # ← ATR vero
df["realized_vol"] = df["Close"].pct_change().rolling(96).std()
df["adx"] = ta.ADX(df["High"], df["Low"], df["Close"], timeperiod=14)          # ← ADX vero
df["vwap"] = vwap_anchored_daily(df)
df.dropna(inplace=True)
print(f"{len(df):,} barre pronte, {df.shape[1]} colonne")
print(f"adx  — min {df['adx'].min():.1f}  mediana {df['adx'].median():.1f}  max {df['adx'].max():.1f}")
print(f"atr  — mediana {df['atr'].median():.6f}  ({df['atr'].median()/df['Close'].median()*100:.3f}% del prezzo)")
df.tail(3)

## Sezione 3 · Dividere i dati in due

Questa è la regola più importante del notebook, e l'unica che non si può violare.

- **In-Sample (80%)** — la parte su cui si cerca, si prova, si sbaglia quante volte si vuole
- **Out-of-Sample (20%)** — la parte che **non si guarda fino alla fine**

Perché conta: su mille combinazioni provate, qualcuna sembrerà eccellente per puro caso — è garantito, anche su dati casuali. L'unico modo per accorgersene è tenere da parte dati che nessuna combinazione ha mai visto.

Se la guardi, cambi qualcosa e riguardi, l'Out-of-Sample non esiste più: è diventata parte della ricerca. **Si apre una volta sola, a ricerca in-sample conclusa** (passo 6 di `ROADMAP_RICERCA.md`).

In [ ]:
IS_RATIO = 0.8
df_is, df_oos = eng.splitting.split_is_oos(df, is_ratio=IS_RATIO)
eng.splitting.describe_split(df_is, df_oos)

## Sezione 4 · Features Engineering

Registra tutti i trigger di ingresso ed uscita nel registro condiviso — va fatto prima di qualunque uso di `event_study` o `exit_search_bt`, che leggono le entry da lì.

In [ ]:
entry_long.registra_trigger_long()
entry_short.registra_trigger_short()
exit_long.registra_exit_long()
exit_short.registra_exit_short()

## SESSIONE 1 · L'andamento del prezzo dopo il trigger

Per ogni condizione: tutte le barre in cui è vera, e dove va il prezzo dopo, fino all'orizzonte `H`. Si entra all'apertura della barra successiva al segnale (mai sullo stesso Close che l'ha generato). La curva è sempre "a favore del trade": per una condizione short, sale quando il prezzo scende.

`H` è una scelta, non una griglia: si calcola una volta sola e tutti gli orizzonti più corti si leggono sulla stessa curva.

**Nessuna occorrenza viene scartata**, nemmeno quelle scattate mentre si era già dentro un trade: contano tutte come eventi indipendenti.

## 1.1 · Le curve a confronto

Ogni linea è una condizione, asse orizzontale le barre dal trigger, verticale la variazione media in percentuale. **La linea nera tratteggiata è il mercato** (nessuna condizione): il metro di paragone — sopra di lei la curva aggiunge qualcosa, appiccicata a lei non dice niente.

Guarda la **forma**: sale e prosegue = il movimento continua; sale e si appiattisce = il vantaggio ha una scadenza (è lì l'orizzonte del trade); sale e ridiscende = quello che guadagnavi lo restituisci.

In [ ]:
H = 12
MIN_TRADES = 200

In [ ]:
ev = eng.event_study.run_event_study(df_is, horizon=H, min_trades=MIN_TRADES)
ev.plot()
#ev.plot_singola("E7_BIG_TAIL_BARS", pips=True)

## 1.2 · La sintesi
| colonna | cosa dice |
|---|---|
| `trades` | quante volte la condizione è scattata |
| `barra_picco` / `barra_picco_netto` | dove la curva è più estrema, e dove si stacca di più dal mercato (usato da `diagnosi()`) |
| `picco_pct` / `picco_pips` | il valore a quel picco — **può essere negativo**: non è un errore, è un trade che perde |
| `incertezza_pct` | quanto balla la media a quel punto (errore standard) |
| `volte_incertezza` | picco diviso incertezza — sotto 2 il picco non si distingue dal rumore |
| `a_fine_pct` | dove sta il prezzo a fine orizzonte: più basso del picco = il movimento è rientrato |
| `vs_mercato_pct` | quanto il trigger si stacca dal movimento che il mercato fa comunque — vicino a zero = è il respiro dell'asset, non il segnale |
| `picco_in_coda` | `True` = il picco netto cade nell'ultimo quarto della finestra, segno di deriva casuale |


**Attenzione al segno:** `picco_pct`/`picco_pips` negativo + backtest sullo stesso trigger che perde = coerente, non un bug.

Perché guardare trades anche qui. MIN_TRADES scarta le condizioni sotto soglia, ma non le rende tutte comparabili: due condizioni appena sopra soglia possono avere un picco grande solo per rumore campionario. Questa tabella è ordinata per picco_pips (ampiezza), non per volte_incertezza (solidità) — prima di scegliere un candidato, controlla sempre la colonna trades e volte_incertezza, non solo la posizione in classifica.

In [ ]:
ev.sintesi.round(4).sort_values('picco_pips', ascending=False).head(5)

### LONG side

In [ ]:
ev.sintesi[ev.sintesi['direction'] == 1][['candidato', 'picco_pips', 'picco_pct']].sort_values('picco_pips', ascending=False).head(6)

### SHORT side

In [ ]:
ev.sintesi[ev.sintesi['direction'] == -1][['candidato', 'picco_pips', 'picco_pct']].sort_values('picco_pips', ascending=False).head(6)

In [ ]:
ev.diagnosi()

# SESSIONE 2 · Le condizioni di uscita

Fin qui abbiamo **misurato** un segnale: nessun capitale, nessuna posizione, nessun costo. Da adesso costruiamo un **sistema**: si sceglie un ingresso e si cerca il modo di uscirne.

## 2.1 · Come vengono calcolati stop e target

Nessuna soglia scritta a mano: uno stop dello 0.15% è enorme su EURUSD e minuscolo su BTCUSD. Per ogni barra si misura quanto sarebbe andato in rosso e in verde un trade lì aperto e tenuto `n` barre; lo stop del trade che apri ora è il **percentile richiesto di quelle escursioni sulle ultime 500 occorrenze concluse** — si allarga da solo su un asset volatile, si stringe quando il mercato si calma. Il valore si fissa all'apertura e non cambia più.

> **Perché "concluse" e non "partite".** L'escursione di un trade aperto alla barra `i` si conosce solo alla `i+n`. Usare le ultime 500 partite userebbe prezzi futuri per uno stop deciso oggi — la finestra è quindi spostata indietro di `n` barre.

**Leva:** parametro `margin` (default `1.0` = nessuna leva). Con leva più alta il motore chiude forzatamente le posizioni se il margine non basta più, così un test a leva alta non mostra profitti che nella realtà non avresti mai visto perché il conto sarebbe stato azzerato prima.

## 2.2 · La classifica

Una riga per combinazione, ordinata per Sharpe. `combinazione` si legge `L=<entry long> · S=<entry short>` (`—` dove quel lato non ha un ingresso). Filtrate di default le combinazioni sotto `min_trades` — se una combinazione attesa non compare, controlla prima lì.

| colonna | cosa dice |
|---|---|
| `trades` | quante volte la condizione è scattata |
| `pnl_pct` | rendimento totale sul periodo |
| `sharpe` | rendimento rapportato alla sua variabilità — è quello che ordina la tabella |
| `max_dd_pct` | la perdita peggiore dal picco precedente |
| `win_rate_pct` | trade in guadagno — da solo non basta: 70% vincenti con perdite doppie dei guadagni è comunque un sistema perdente |
| `profit_factor` | incasso dei vincenti per ogni euro perso dai perdenti: sotto 1 il sistema perde |
| `avg_trade` | guadagno medio per trade, in pips — il numero da confrontare col costo round-turn |
| `durata_media` / `durata_max` | durata dei trade, in media e nel caso peggiore |

**A cosa serve confrontare le righe:** in una chiamata l'uscita (`n_barre`, `perc_sl`, `perc_tp`) è identica per tutte le combinazioni — la tabella risponde a "quale entry rende di più con QUESTA uscita", non "quale uscita è la migliore". Per confrontare uscite diverse si rilancia la funzione con altri valori e si mettono le tabelle a confronto a mano.

## 2.3 · Backtest con i trigger selezionati

Griglia completa su tutte le entry sopravvissute/selezionate alla Sessione 1.

**TEST 1**

Condizioni di uscita: STOP LOSS, TAKE PROFIT

In [ ]:
es = run_exit_search_bt(df_is, entry_cols_long=["E14_HARAMI_CROSS_CONFIRMED", None],
                        entry_cols_short=["E4_SHORT_EMA_CROSS_DOWN", None],
                        n_barre=12, perc_sl=90.0, perc_tp=0.0,
                        commission=0.0,)
es.top(10)
#es.trades("L=E7_BIG_TAIL_BARS · S=E9_SHORT_CLOSING_PATTERN_ONLY_II")

**TEST 2**

Condizioni di uscita: STOP LOSS, TAKE PROFIT, Regole meccaniche

In [ ]:
es = run_exit_search_bt(df_is,
                        entry_cols_long=["E14_HARAMI_CROSS_CONFIRMED"],
                        entry_cols_short=["E4_SHORT_EMA_CROSS_DOWN"],
                        n_barre=12,
                        exit_rule_pairs=["RSI_EXTREME", "MACD_CROSS",
                                         "EMA_CROSS", None],
                        perc_sl=90.0, perc_tp=0.0, commission=0.0,
                        min_trades=30,)
es.top(10)